In [ ]:

!pip install -q safetensors huggingface_hub openpyxl tqdm


In [ ]:
# ============================================================
# 2. IMPORT
# ============================================================

import os
import json
import re
from pathlib import Path
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torchvision.models import mobilenet_v2

import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    auc,
)
from sklearn.preprocessing import label_binarize

from safetensors.torch import load_file
from huggingface_hub import hf_hub_download

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
# ============================================================
# 3. CONFIG 
# ============================================================

CLASS_NAMES = ["nv", "mel", "bkl", "bcc", "akiec", "vasc", "df"]
LABEL_MAP = {name: idx for idx, name in enumerate(CLASS_NAMES)}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TEST_CSV = "/kaggle/input/datasets/chisdong/global-testv2/isic2019_external_global_test_like_ham10000.csv"

IMAGE_DIRS = [
    "/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input"
]

OUTPUT_DIR = Path("/kaggle/working/batch_eval_results")

BATCH_SIZE = 32
NUM_WORKERS = 2
IMAGE_SIZE = 224

HF_TOKEN = os.environ.get("HF_TOKEN", None)

print("DEVICE:", DEVICE)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# ============================================================
# 4. MODEL PATHS - CHỈ CẦN SỬA LIST NÀY
# ============================================================

MODEL_PATHS = [
    # IID
    {
        "path": "ChisDong/ServerFL_IID/global_model_round_23.safetensors",
        "scenario": "IID",
        "method": "FedAvg",
        "display_name": "IID - FedAvg"
    },
    {
        "path": "ChisDong/ServerFL_IID/fedprox/global_model_round_21.safetensors",
        "scenario": "IID",
        "method": "FedProx",
        "display_name": "IID - FedProx"
    },
    {
        "path": "ChisDong/ServerFL_IID/cw_risk_ucfedavg/global_model_round_18.safetensors",
        "scenario": "IID",
        "method": "CW-Risk-UCFedAvg",
        "display_name": "IID - CW-Risk-UCFedAvg"
    },

    # Non-IID
    {
        "path": "ChisDong/ServerFL_nonIID/global_model_round_21.safetensors",
        "scenario": "Non-IID",
        "method": "FedAvg",
        "display_name": "Non-IID - FedAvg"
    },
    {
        "path": "ChisDong/ServerFL_nonIID/fedprox/global_model_round_20.safetensors",
        "scenario": "Non-IID",
        "method": "FedProx",
        "display_name": "Non-IID - FedProx"
    },
    {
        "path": "ChisDong/ServerFL_nonIID/cw_risk_ucfedavg/global_model_round_6.safetensors",
        "scenario": "Non-IID",
        "method": "CW-Risk-UCFedAvg",
        "display_name": "Non-IID - CW-Risk-UCFedAvg"
    },

    # Quantity-skew
    {
        "path": "ChisDong/ServerFL_quantity/global_model_round_13.safetensors",
        "scenario": "Quantity-skew",
        "method": "FedAvg",
        "display_name": "Quantity-skew - FedAvg"
    },
    {
        "path": "ChisDong/ServerFL_quantity/fedprox/global_model_round_15.safetensors",
        "scenario": "Quantity-skew",
        "method": "FedProx",
        "display_name": "Quantity-skew - FedProx"
    },
    {
        "path": "ChisDong/ServerFL_quantity/cw_risk_ucfedavg/global_model_round_16.safetensors",
        "scenario": "Quantity-skew",
        "method": "CW-Risk-UCFedAvg",
        "display_name": "Quantity-skew - CW-Risk-UCFedAvg"
    },
]


In [ ]:
# ============================================================
# 5. PARSE MODEL PATHS 
# ============================================================

def parse_model_path(item) -> Dict:
    """
    Hỗ trợ 2 kiểu:
    1. Dictionary có path, scenario, method, display_name
    2. String path cũ, fallback nếu cần
    """

    # Trường hợp mới: tự đặt tên bằng dictionary
    if isinstance(item, dict):
        model_path = str(item["path"]).strip()

        scenario = item.get("scenario", "Unknown")
        method = item.get("method", "Unknown")
        display_name = item.get("display_name", f"{scenario} - {method}")

    # Trường hợp cũ: chỉ truyền string path
    else:
        model_path = str(item).strip()
        scenario = "Unknown"
        method = Path(model_path).stem
        display_name = method

    # Local Kaggle path
    if model_path.startswith("/kaggle/") or model_path.startswith("/mnt/") or model_path.startswith("./") or model_path.startswith("../"):
        return {
            "raw_path": model_path,
            "source": "local",
            "repo_id": "",
            "filename": "",
            "local_path": model_path,
            "method": method,
            "scenario": scenario,
            "display_name": display_name,
        }

    # Hugging Face path
    parts = model_path.split("/")

    if len(parts) < 3:
        raise ValueError(f"HF path không hợp lệ: {model_path}")

    repo_id = "/".join(parts[:2])
    filename = "/".join(parts[2:])

    return {
        "raw_path": model_path,
        "source": "hf",
        "repo_id": repo_id,
        "filename": filename,
        "local_path": "",
        "method": method,
        "scenario": scenario,
        "display_name": display_name,
    }


model_rows = [parse_model_path(p) for p in MODEL_PATHS]
models_df = pd.DataFrame(model_rows)

models_df

In [ ]:
# ============================================================
# 6. DATASET CLASS
# ============================================================

class SkinLesionDataset(Dataset):
    def __init__(self, csv_file: str, image_dirs: List[str], transform=None):
        self.df = pd.read_csv(csv_file)
        self.image_dirs = image_dirs
        self.transform = transform

        self._validate_columns()
        self.image_path_map = self._build_image_path_map()
        self._validate_images_exist()

        print(f"[DATASET] CSV: {csv_file}")
        print(f"[DATASET] Samples: {len(self.df)}")
        print(f"[DATASET] Indexed images: {len(self.image_path_map)}")
        print("[DATASET] Class distribution:")
        print(self.df["dx"].value_counts())

    def _validate_columns(self):
        required = {"image_id", "dx"}
        missing = required - set(self.df.columns)

        if missing:
            raise ValueError(f"CSV thiếu cột bắt buộc: {missing}. Cần image_id và dx.")

        unknown = set(self.df["dx"].astype(str)) - set(CLASS_NAMES)
        if unknown:
            raise ValueError(f"CSV có label không thuộc CLASS_NAMES: {unknown}")

    def _build_image_path_map(self) -> Dict[str, str]:
        path_map = {}

        for image_dir in self.image_dirs:
            image_dir = str(image_dir)

            if not os.path.exists(image_dir):
                print(f"[WARN] Image dir không tồn tại: {image_dir}")
                continue

            for root, _, files in os.walk(image_dir):
                for fname in files:
                    lower = fname.lower()

                    if lower.endswith((".jpg", ".jpeg", ".png")):
                        stem = os.path.splitext(fname)[0]
                        full_path = os.path.join(root, fname)
                        path_map[stem] = full_path
                        path_map[fname] = full_path

        return path_map

    def _validate_images_exist(self):
        missing = []

        for image_id in self.df["image_id"].astype(str).values:
            if image_id not in self.image_path_map:
                stem = os.path.splitext(image_id)[0]

                if stem not in self.image_path_map:
                    missing.append(image_id)

        if missing:
            print(f"[WARN] Missing images: {len(missing)}")
            print("[WARN] First missing:", missing[:10])

            raise FileNotFoundError(
                f"Không tìm thấy {len(missing)} ảnh. Kiểm tra TEST_CSV và IMAGE_DIRS."
            )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_id = str(row["image_id"])
        label_name = str(row["dx"])

        if image_id in self.image_path_map:
            image_path = self.image_path_map[image_id]
        else:
            stem = os.path.splitext(image_id)[0]
            image_path = self.image_path_map[stem]

        image = Image.open(image_path).convert("RGB")
        label = LABEL_MAP[label_name]

        if self.transform:
            image = self.transform(image)

        return image, label


In [ ]:
# ============================================================
# 7. MODEL LOADER
# ============================================================

def build_model(num_classes: int = 7):
    model = mobilenet_v2(weights="DEFAULT")
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model


def load_model_from_row(row: pd.Series, device: str, hf_token: Optional[str] = None):
    method = str(row.get("method", "unknown"))
    source = str(row.get("source", ""))

    if source == "local":
        model_path = str(row["local_path"])

        if not os.path.exists(model_path):
            raise FileNotFoundError(f"[{method}] local_path không tồn tại: {model_path}")

    elif source == "hf":
        repo_id = str(row["repo_id"])
        filename = str(row["filename"])

        model_path = hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            repo_type="model",
            token=hf_token,
        )

    else:
        raise ValueError(f"Unknown source: {source}")

    state_dict = load_file(model_path)

    model = build_model(num_classes=len(CLASS_NAMES))
    model.load_state_dict(state_dict, strict=True)
    model = model.to(device)
    model.eval()

    return model, model_path


In [ ]:
# ============================================================
# 8. METRICS
# ============================================================

def compute_specificity_macro(y_true: np.ndarray, y_pred: np.ndarray, num_classes: int):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    specificities = []

    for i in range(num_classes):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp

        denom = tn + fp
        spe = tn / denom if denom > 0 else 0.0
        specificities.append(spe)

    return float(np.mean(specificities)), specificities


def safe_auc_macro_weighted(y_true, y_score, num_classes):
    try:
        auc_macro = roc_auc_score(
            y_true,
            y_score,
            multi_class="ovr",
            average="macro",
            labels=list(range(num_classes)),
        )
    except Exception:
        auc_macro = np.nan

    try:
        auc_weighted = roc_auc_score(
            y_true,
            y_score,
            multi_class="ovr",
            average="weighted",
            labels=list(range(num_classes)),
        )
    except Exception:
        auc_weighted = np.nan

    return auc_macro, auc_weighted


def compute_per_class_auc(y_true, y_score, num_classes):
    y_true_bin = label_binarize(y_true, classes=list(range(num_classes)))
    per_auc = {}

    for i, class_name in enumerate(CLASS_NAMES):
        positives = int(np.sum(y_true_bin[:, i] == 1))
        negatives = int(np.sum(y_true_bin[:, i] == 0))

        if positives == 0 or negatives == 0:
            per_auc[class_name] = np.nan
            continue

        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        per_auc[class_name] = float(auc(fpr, tpr))

    return per_auc


def evaluate_model(model, loader, device: str):
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Predicting", leave=False):
            images = images.to(device)

            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    y_score = np.array(all_probs)

    num_classes = len(CLASS_NAMES)

    acc = accuracy_score(y_true, y_pred)
    precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

    specificity_macro, specificity_per_class = compute_specificity_macro(
        y_true, y_pred, num_classes
    )

    auc_macro, auc_weighted = safe_auc_macro_weighted(y_true, y_score, num_classes)
    per_class_auc = compute_per_class_auc(y_true, y_score, num_classes)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)

    per_class_precision = precision_score(
        y_true, y_pred, average=None, labels=list(range(num_classes)), zero_division=0
    )
    per_class_recall = recall_score(
        y_true, y_pred, average=None, labels=list(range(num_classes)), zero_division=0
    )
    per_class_f1 = f1_score(
        y_true, y_pred, average=None, labels=list(range(num_classes)), zero_division=0
    )

    optimization_score = (
        0.40 * (0.0 if np.isnan(auc_macro) else auc_macro)
        + 0.30 * recall_macro
        + 0.20 * f1_macro
        + 0.10 * specificity_macro
    )

    result = {
        "y_true": y_true,
        "y_pred": y_pred,
        "y_score": y_score,
        "metrics": {
            "accuracy": float(acc),
            "precision_macro": float(precision_macro),
            "recall_macro": float(recall_macro),
            "specificity_macro": float(specificity_macro),
            "f1_macro": float(f1_macro),
            "auc_macro_ovr": float(auc_macro) if not np.isnan(auc_macro) else None,
            "auc_weighted_ovr": float(auc_weighted) if not np.isnan(auc_weighted) else None,
            "optimization_score": float(optimization_score),
        },
        "per_class": {},
        "confusion_matrix": cm.tolist(),
        "normalized_confusion_matrix": cm_norm.tolist(),
    }

    for i, class_name in enumerate(CLASS_NAMES):
        result["per_class"][class_name] = {
            "precision": float(per_class_precision[i]),
            "recall": float(per_class_recall[i]),
            "specificity": float(specificity_per_class[i]),
            "f1": float(per_class_f1[i]),
            "auc": None if np.isnan(per_class_auc[class_name]) else float(per_class_auc[class_name]),
            "support": int(np.sum(y_true == i)),
        }

    return result


In [ ]:
# ============================================================
# 9. PLOTS
# ============================================================

def plot_normalized_confusion_matrix(cm_norm, method_name: str, output_path: Path):
    cm_norm = np.array(cm_norm)

    plt.figure(figsize=(8, 6))
    im = plt.imshow(cm_norm, interpolation="nearest", cmap="Blues")
    plt.title(f"Normalized Confusion Matrix of {method_name}")
    plt.colorbar(im, fraction=0.046, pad=0.04)

    tick_marks = np.arange(len(CLASS_NAMES))
    plt.xticks(tick_marks, CLASS_NAMES, rotation=45, ha="right", fontsize=9)
    plt.yticks(tick_marks, CLASS_NAMES, fontsize=9)

    plt.xlabel("Predicted label")
    plt.ylabel("True label")

    threshold = cm_norm.max() / 2.0

    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            value = cm_norm[i, j]
            plt.text(
                j, i, f"{value:.2f}",
                ha="center",
                va="center",
                color="white" if value > threshold else "black",
                fontsize=8,
            )

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_roc_auc(y_true, y_score, method_name: str, auc_macro, auc_weighted, output_path: Path):
    y_true_bin = label_binarize(y_true, classes=list(range(len(CLASS_NAMES))))

    plt.figure(figsize=(10, 8))

    for i, class_name in enumerate(CLASS_NAMES):
        positives = np.sum(y_true_bin[:, i] == 1)
        negatives = np.sum(y_true_bin[:, i] == 0)

        if positives == 0 or negatives == 0:
            continue

        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{class_name} AUC = {roc_auc:.4f}")

    plt.plot([0, 1], [0, 1], "k--", label="Random Guess")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate / Sensitivity")

    auc_macro_text = "nan" if auc_macro is None else f"{auc_macro:.4f}"
    auc_weighted_text = "nan" if auc_weighted is None else f"{auc_weighted:.4f}"

    plt.title(
        f"{method_name} ROC-AUC Curve - One-vs-Rest\n"
        f"Macro AUC={auc_macro_text}, Weighted AUC={auc_weighted_text}"
    )
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_metric_comparison(summary_df: pd.DataFrame, output_path: Path):
    metrics = ["accuracy", "recall_macro", "f1_macro", "auc_macro_ovr", "auc_weighted_ovr"]

    plot_df = summary_df[["scenario", "method"] + metrics].copy()
    plot_df["label"] = plot_df["scenario"].astype(str) + "_" + plot_df["method"].astype(str)
    plot_df = plot_df.set_index("label")[metrics]

    ax = plot_df.plot(kind="bar", figsize=(14, 6))
    ax.set_ylim(0, 1)
    ax.set_title("Model Comparison")
    ax.set_ylabel("Score")
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_per_class_auc(per_class_df: pd.DataFrame, output_path: Path):
    df = per_class_df.copy()
    df["label"] = df["scenario"].astype(str) + "_" + df["method"].astype(str)

    pivot = df.pivot(index="class", columns="label", values="auc")
    pivot = pivot.loc[CLASS_NAMES]

    ax = pivot.plot(kind="bar", figsize=(14, 6))
    ax.set_ylim(0, 1)
    ax.set_title("Per-class AUC Comparison")
    ax.set_ylabel("AUC")
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()


In [ ]:
# ============================================================
# 10. CHUẨN BỊ DATA LOADER
# ============================================================

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

test_dataset = SkinLesionDataset(
    csv_file=TEST_CSV,
    image_dirs=IMAGE_DIRS,
    transform=test_transform,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

len(test_dataset)


In [ ]:
HF_TOKEN = None

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    print("[HF] Token loaded from Kaggle Secrets.")
except Exception:
    print("[HF] No Kaggle secret loaded. If repo is public, this is OK.")

In [ ]:
# ============================================================
# 12. LƯU KẾT QUẢ RA FILE
# ============================================================

summary_csv = OUTPUT_DIR / "results_summary.csv"
per_class_csv = OUTPUT_DIR / "results_per_class.csv"
json_path = OUTPUT_DIR / "all_metrics.json"
excel_path = OUTPUT_DIR / "results_summary.xlsx"

summary_df.to_csv(summary_csv, index=False)
per_class_df.to_csv(per_class_csv, index=False)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, ensure_ascii=False, indent=4)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="summary", index=False)
    per_class_df.to_excel(writer, sheet_name="per_class", index=False)

plot_metric_comparison(summary_df, plots_dir / "overall_metric_comparison.png")
plot_per_class_auc(per_class_df, plots_dir / "per_class_auc_comparison.png")

print("[DONE] Batch evaluation completed.")
print("Summary CSV:", summary_csv)
print("Per-class CSV:", per_class_csv)
print("Excel:", excel_path)
print("JSON:", json_path)
print("Plots dir:", plots_dir)
print("Predictions dir:", predictions_dir)


In [ ]:
# ============================================================
# 11. CHẠY BATCH EVALUATION
# ============================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plots_dir = OUTPUT_DIR / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

predictions_dir = OUTPUT_DIR / "predictions"
predictions_dir.mkdir(parents=True, exist_ok=True)

summary_rows = []
per_class_rows = []
all_metrics = {}

for idx, row in models_df.iterrows():
    method = str(row["method"])
    scenario = str(row.get("scenario", ""))
    display_name = str(row.get("display_name", f"{scenario} - {method}"))
    
    safe_method = display_name.replace(" ", "_").replace("/", "_").replace("-", "_")
    print("\n" + "=" * 80)
    print(f"[MODEL {idx + 1}/{len(models_df)}] {display_name}")
    print("Raw path:", row["raw_path"])
    print("=" * 80)

    model, downloaded_model_path = load_model_from_row(row, device=DEVICE, hf_token=HF_TOKEN)
    print(f"[INFO] Loaded model: {downloaded_model_path}")

    result = evaluate_model(model, test_loader, device=DEVICE)
    metrics = result["metrics"]

    summary_row = {
        "scenario": scenario,
        "method": method,
        "display_name": display_name,
        "raw_path": row["raw_path"],
        "downloaded_model_path": downloaded_model_path,
        **metrics,
    }
    summary_rows.append(summary_row)

    for class_name, class_metrics in result["per_class"].items():
        per_class_rows.append({
            "scenario": scenario,
            "method": method,
            "display_name": display_name,
            "raw_path": row["raw_path"],
            "class": class_name,
            **class_metrics,
        })

    pred_df = pd.DataFrame({
        "y_true": result["y_true"],
        "y_true_label": [CLASS_NAMES[int(x)] for x in result["y_true"]],
        "y_pred": result["y_pred"],
        "y_pred_label": [CLASS_NAMES[int(x)] for x in result["y_pred"]],
    })

    for i, class_name in enumerate(CLASS_NAMES):
        pred_df[f"prob_{class_name}"] = result["y_score"][:, i]

    pred_path = predictions_dir / f"{safe_method}_predictions.csv"
    pred_df.to_csv(pred_path, index=False)

    cm_path = plots_dir / f"{safe_method}_normalized_confusion_matrix.png"
    roc_path = plots_dir / f"{safe_method}_roc_auc_curve.png"

    plot_normalized_confusion_matrix(
        result["normalized_confusion_matrix"],
        method_name= display_name,
        output_path=cm_path,
    )

    plot_roc_auc(
        y_true=result["y_true"],
        y_score=result["y_score"],
        method_name= display_name,
        auc_macro=metrics["auc_macro_ovr"],
        auc_weighted=metrics["auc_weighted_ovr"],
        output_path=roc_path,
    )

    key = display_name

    all_metrics[key] = {
        "scenario": scenario,
        "method": method,
        "raw_path": row["raw_path"],
        "downloaded_model_path": downloaded_model_path,
        "metrics": metrics,
        "per_class": result["per_class"],
        "confusion_matrix": result["confusion_matrix"],
        "normalized_confusion_matrix": result["normalized_confusion_matrix"],
        "plots": {
            "normalized_confusion_matrix": str(cm_path),
            "roc_auc_curve": str(roc_path),
        },
        "predictions": str(pred_path),
    }

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

summary_df = pd.DataFrame(summary_rows)
per_class_df = pd.DataFrame(per_class_rows)

if "optimization_score" in summary_df.columns:
    summary_df = summary_df.sort_values("optimization_score", ascending=False)

summary_df


In [ ]:
# ============================================================
# 13. XEM BẢNG TỔNG HỢP
# ============================================================

summary_df


In [ ]:
# ============================================================
# 14. XEM BẢNG PER-CLASS AUC DẠNG PIVOT
# ============================================================

auc_pivot = per_class_df.pivot_table(
    index=["scenario", "method"],
    columns="class",
    values="auc"
).reset_index()

auc_pivot


In [ ]:
# ============================================================
# 15. NÉN KẾT QUẢ ĐỂ DOWNLOAD
# ============================================================

!cd /kaggle/working && zip -r batch_eval_results.zip batch_eval_results

print("Download file:")
print("/kaggle/working/batch_eval_results.zip")
